In [ ]:
import os
os.environ['OPENAI_API_KEY'] =''

In [ ]:
from langchain.vectorstores import Chroma
from langchain.text_splitter import CharacterTextSplitter
from langchain.chains.question_answering import load_qa_chain
from langchain.chains import VectorDBQA
from langchain.llms import OpenAI
import openai
import time
from langchain.document_loaders import DirectoryLoader
from langchain.chains import RetrievalQA
from langchain.chains.qa_with_sources import load_qa_with_sources_chain
from langchain.embeddings.openai import OpenAIEmbeddings

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

## Splitting Documents into chunks

In [ ]:
loader=DirectoryLoader('')
docs=loader.load()
character_text_splitter=CharacterTextSplitter(separator='.',chunk_size=1000,chunk_overlap=250)
doc_texts=character_text_splitter.split_documents(docs)


In [ ]:
len(doc_texts)

## Legal Embedding Creation and Storing

In [ ]:
persist_directory = ''
embeddings=OpenAIEmbeddings(openai_api_key=os.environ['OPENAI_API_KEY'],model='text-embedding-ada-002')

In [ ]:
vectordb=Chroma.from_documents(documents=doc_texts[0:1001], embedding=embeddings, persist_directory=persist_directory)

In [ ]:
start=1001
end=2001
break_at=len(doc_texts)
while(True):
    vectordb=Chroma.from_documents(documents=doc_texts[start:end], embedding=embeddings, persist_directory=persist_directory)
    start=end
    end=end+1000
    print(start,end)
    if end>break_at:
        vectordb=Chroma.from_documents(documents=doc_texts[start:], embedding=embeddings, persist_directory=persist_directory)
        break
    time.sleep(20)

In [ ]:
vectordb.persist()

In [ ]:
vectordb=None

In [ ]:
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embeddings)

In [ ]:
llm=OpenAI(temperature=0,openai_api_key=os.environ['OPENAI_API_KEY'],model_name='text-davinci-003')
chain=load_qa_chain(llm,chain_type='refine')

In [ ]:
prompt='''Give relevant answer for the question from the sources given in the document specific to India.Give answer like humans.Be interactive while answering.The answer shoudld be complete.If you dont know the answer,just say Sorry,I dont know.
    Question:
'''

In [ ]:
query='''I hacked a friend's Instagram as a prank to show him later. But he is taking it too seriously. If he presses charges, what are my options?'''
context_docs=vectordb.similarity_search(query)

In [ ]:
result=chain.run(input_documents=context_docs,question=prompt+query)
print(result)

In [ ]:
ground_truth=pd.read_csv('')

In [ ]:
ground_truth=ground_truth.drop(columns=['Unnamed: 0'])

In [ ]:
ground_truth.head(3)

In [ ]:
result=pd.DataFrame(columns=['title','question','ground_truth','answer_generated','score'])

In [ ]:
rows=[]

for i in tqdm(range(len(ground_truth))):
    query='Title:'+ground_truth['title'][i].strip()+'\n'+'Question:'+ground_truth['question'][i].strip()
    context_docs=vectordb.similarity_search(query)
    result=chain.run(input_documents=context_docs,question=prompt+query)
    rows.append([ground_truth['title'][i],ground_truth['question'][i],ground_truth['answer'][i],result.strip(),0])
    time.sleep(10)

In [ ]:
len(rows)

In [ ]:
import csv
fields=['title','question','ground_truth','answer_generated','score']
with open('', 'w+') as f:
      
    # using csv.writer method from CSV package
    write = csv.writer(f)
      
    write.writerow(fields)
    write.writerows(rows)
f.close()